In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
import pandas as pd
import random

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from huggingface_hub import login
from google.colab import userdata

In [3]:
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [4]:
HF_TOKEN = userdata.get('HF')
login(token=HF_TOKEN)

In [5]:
df = pd.read_csv('/content/drive/MyDrive/LR1.csv')
df_dev = pd.read_csv('/content/drive/MyDrive/LR1_dev.csv')
df_answers = pd.read_csv('/content/drive/MyDrive/LR1_dev_answers.csv')

In [ ]:
category_counts = df['category'].value_counts()
print(f"Категории: {category_counts.index.tolist()}")

In [7]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model_id = "Qwen/Qwen2.5-7B-Instruct"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
!pip uninstall -y bitsandbytes

In [6]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.8 MB/s eta 0:00:00


In [ ]:
import os
os._exit(0)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [11]:
MATH_PROMPT = """You are an expert in the exact sciences. Remember the theory, use engineering approach, calculate in detail, solve the problem step by step and choose the correct answer (count the index starting from 0). Think in english."""

LAW_PROMPT = """You are an expert in the humanities. Analyze the situation and choose the correct answer (count the index from 0). Think in english."""

SCIENCE_PROMPT = """You are an expert in natural sciences. Remember the facts and choose the correct answer (count the index from 0). Think in english."""

BUSINESS_PROMPT = """You are an expert in applied disciplines. Solve the problem step by step and choose the correct answer (count the index starting from 0). Think in english."""

DEFAULT_PROMPT = """You are a universal expert. Solve the problem step by step and choose the correct answer (count the index from 0). Think in english."""

CATEGORY_TO_PROMPT = {
    'math': MATH_PROMPT, 'physics': MATH_PROMPT, 'economics': MATH_PROMPT, 'engineering': MATH_PROMPT,
    'law': LAW_PROMPT, 'philosophy': LAW_PROMPT, 'psychology': LAW_PROMPT,
    'biology': SCIENCE_PROMPT, 'chemistry': SCIENCE_PROMPT, 'health': SCIENCE_PROMPT,
    'business': BUSINESS_PROMPT, 'computer science': BUSINESS_PROMPT, 'history': BUSINESS_PROMPT,
    'other': DEFAULT_PROMPT
}

USER_PROMPT = """
Задача: {question}
Варианты: {options}
"""

In [12]:
!pip install langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.3/566.3 kB 23.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.3.4
    Uninstalling huggingface_hub-1.3.4:
      Successfully uninstalled huggingface_hub-1.3.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.1 which is incompatible.


In [13]:
from transformers import pipeline, GenerationConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.messages import HumanMessage, SystemMessage

In [14]:
import re
import ast

In [15]:
def format_options(options_str):
    options = re.findall(r"'([^']+)'", options_str)
    return "\n".join([f"{i}: {opt}" for i, opt in enumerate(options)])

In [16]:
def build_qwen_prompt(system_message, user_message):
    prompt = f"""<|im_start|>system
{system_message}<|im_end|>
<|im_start|>user
{user_message}<|im_end|>
<|im_start|>assistant
"""
    return prompt

In [17]:
def stage1_generate_prompt(category):
  system_msg = "Ты создаешь системные промпты для решения задач с помощью Qwen/Qwen2.5-7B-Instruct. Будь кратким, конкретным и пиши на русском языке."

  user_msg = f"""Категория: {category}

Создай эффективный системный промпт для модели, который:
1. Определит роль модели (эксперт в конкретной области)
2. Укажет конкретную инструкцию(анализ, решение, выбор варианта)
3. Поможет выбрать правильный вариант ответа (индекс ответа выбирается, начиная с 0) для предоставленного вопроса.
4. Включит специфические требования для этой категории задач
5. Будет кратким (1-2 предложения)

Примеры системных сообщений:
Ты эксперт по математике. Реши задачу шаг за шагом и выбери номер правильного ответа. Отвечай только цифрой.
Ты юрист. Проанализируй правовую ситуацию, определи применимые нормы и выбери номер корректного варианта. Ответ - только число.
Ты senior разработчик. Проанализируй код, найди ошибку и выбери номер правильного решения. Ответь только цифрой.

Промпт:"""

  prompt = build_qwen_prompt(system_msg, user_msg)
  return prompt

In [18]:
def stage2_solve_with_prompt(custom_prompt, question, options):
  system_msg = custom_prompt

  user_msg = f"""Task: {question}

Options:
{options}

Choose the correct answer (start with 0)."""
  prompt = build_qwen_prompt(system_msg, user_msg)
  return prompt

In [19]:
def stage3_extract_final_answer(analysis, options):
    system_msg = "You extract the answer number from the text. Answer only with number."

    user_msg = f"""From the text below, extract the number of the correct version (the response index is selected starting from 0).
If the number is explicitly specified, return it. If not - define by context.

Options: {options}

Text: {analysis}

Answer number:"""

    prompt = build_qwen_prompt(system_msg, user_msg)
    return prompt

In [20]:
GENERATOR = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer
    )

In [21]:
def process_question(row):
    category = row['category']
    question = row['question']
    options = format_options(row['options'])
    print(options)
    q_id = row['Unnamed: 0']

    if category not in CATEGORY_TO_PROMPT:
      category = 'other'
    custom = CATEGORY_TO_PROMPT[category]

    GENERATOR.generation_config.max_new_tokens = 1024
    GENERATOR.generation_config.temperature = 0.2
    GENERATOR.generation_config.repetition_penalty = 1.05
    GENERATOR.generation_config.top_p = 0.9
    GENERATOR.generation_config.do_sample = True

    prompt2 = stage2_solve_with_prompt(custom, question, options)

    result2 = GENERATOR(prompt2)
    response2 = result2[0]['generated_text']
    analysis = response2.split("<|im_start|>assistant")[-1].strip()
    print(analysis)

    GENERATOR.generation_config.max_new_tokens = 50
    GENERATOR.generation_config.do_sample = False

    prompt3 = stage3_extract_final_answer(analysis, options)
    result3 = GENERATOR(prompt3)
    response3 = result3[0]['generated_text']
    answer = response3.split("<|im_start|>assistant")[-1].strip()

    return answer

In [22]:
res = {'answer': []}
for idx, row in df.iterrows():
  answer = process_question(row)
  res['answer'].append(answer)
df_an = pd.DataFrame(res)
df_an.to_csv('answers.csv')

0: недопустимые, потому что показания полицейского могут быть признаны предвзятыми.
1: допустимые, потому что мужчина добровольно признался.
2: допустимые, потому что мужчине зачитали права Миранды перед тем, как было получено признание.
3: допустимые, потому что мужчина был осведомлен о своих правах, когда признался.
4: недопустимые, потому что во время признания мужчине не предоставили адвоката.
5: недопустимые, потому что видеозапись является наилучшим доказательством признания мужчины.
6: допустимые, потому что признание было записано на видео.
7: недопустимые, потому что это косвенные доказательства, не попадающие ни под одно из признанных исключений.
8: допустимые, потому что полицейский был свидетелем признания.
9: недопустимые, потому что мужчина находился под давлением во время признания.


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The correct answer is 2: допустимые, потому что мужчине зачитали права Миранды перед тем, как было получено признание.

Explanation: In the United States, the Miranda rights include informing a suspect of their right to remain silent and their right to an attorney before they can be questioned by police. If these rights are properly read and understood by the suspect, any statements made after that point can generally be considered voluntary and admissible in court. Since the passage states that the man was informed of his rights before making the confession, this makes the confession admissible under standard legal procedures.


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0: экзистенциалистской
1: идеологической
2: апатичной
3: агностической
4: светской
5: духовной
6: религиозной
7: философской
8: прагматической
9: нерелигиозной


...

The most common expected side effect of creatine supplementation is muscle cramping or discomfort, which corresponds to index 6. Therefore, the correct answer is 6.


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


0: Заказов покупателя до журнала продаж.
1: Транспортных документов до журнала поступлений наличных.
2: Журнала поступлений наличных до транспортных документов.
3: Журнала продаж до журнала поступлений наличных.
4: Журнала продаж до транспортных документов.
5: Транспортных документов до журнала продаж.
6: Заказов покупателя до журнала поступлений наличных.
7: Журнала поступлений наличных до заказов покупателя.
8: Транспортных документов до заказов покупателя.
9: Заказов покупателя до транспортных документов.


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To solve this problem, let's break it down step by step:

1. The task is to identify the correct sequence of documents for an auditor to follow when checking for inflated sales volumes.
2. The goal is to detect overstatement of sales volume.
3. In accounting practices, the typical flow of documents for sales transactions is:
   - Customer orders (or purchase orders)
   - Sales invoices
   - Shipping documents (transport documents)
   - Journal entries for sales (journal of sales)
   - Cash receipts journal (journal of cash receipts)

4. To check for inflated sales, the auditor would start at the point where sales are recorded and trace back to where the initial sales orders were made.

Given these steps, the logical sequence should be from the journal of sales (where final sales are recorded) back to the customer orders (where the initial sales requests are made).

Therefore, the correct answer is:

9: Orders from customers to transport documents.

This path allows the auditor to verif

In [ ]:
correct_rows, rows = 0, 0
for idx, row in df_dev.iterrows():
    answer = process_question(row)
    correct_row = df_answers.iloc[idx]['answer']
    print(answer, correct_row)
    rows += 1
    if int(answer) == int(correct_row):
      correct_rows += 1
print(f'{correct_rows / rows * 100}%')